<a href="https://colab.research.google.com/github/Chrisisnot-cyber/AI-Search-Assistant/blob/main/AI_Research_Assistant_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q openai duckduckgo-search langchain tiktoken gradio nltk bs4 requests transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 92.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 6.1 MB/s eta 0:00:00


In [3]:
import os
import re
import json
import requests
import numpy as np
from typing import List, Dict, Any, Optional, Tuple
from dataclasses import dataclass
from bs4 import BeautifulSoup
import nltk
from nltk.tokenize import sent_tokenize
import tiktoken
from datetime import datetime
import gradio as gr

# Download NLTK data for sentence tokenization
nltk.download('punkt', quiet=True)

# Set up OpenAI API key from Google Colab secrets
from google.colab import userdata

# Configure OpenAI client
from openai import OpenAI

# Set up encoding for token counting
encoding = tiktoken.get_encoding("cl100k_base")

# Constants
MAX_TOKENS_SUMMARY = 4096
MAX_TOKENS_CONTEXT = 8192
TOP_SEARCH_RESULTS = 8
TRUSTED_DOMAINS = [".gov", ".edu", ".org", "reuters.com", "nature.com", "science.org",
                   "bbc.com", "nytimes.com", "washingtonpost.com", "economist.com"]


In [4]:
# Enhanced web search module
from duckduckgo_search import DDGS
import requests
from bs4 import BeautifulSoup

@dataclass
class SearchResult:
    """Data class to store search result information"""
    title: str
    body: str
    href: str
    date: Optional[str] = None
    content: str = ""

    def __post_init__(self):
        # Clean up text
        self.title = re.sub(r'\s+', ' ', self.title).strip()
        self.body = re.sub(r'\s+', ' ', self.body).strip()

    def fetch_content(self, timeout: int = 10) -> bool:
        """Fetch the full content of the article"""
        try:
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            }
            response = requests.get(self.href, headers=headers, timeout=timeout)
            response.raise_for_status()

            soup = BeautifulSoup(response.text, 'html.parser')

            # Remove script, style elements and comments
            for element in soup(['script', 'style', 'header', 'footer', 'nav']):
                element.decompose()

            # Extract main content based on common article containers
            article = soup.find('article') or soup.find(class_=re.compile(r'article|content|main|post'))

            if article:
                text = article.get_text(separator=' ', strip=True)
            else:
                # Fallback to main content area
                main = soup.find('main') or soup.find(attrs={'role': 'main'})
                if main:
                    text = main.get_text(separator=' ', strip=True)
                else:
                    # Last resort: get body text
                    text = soup.body.get_text(separator=' ', strip=True)

            # Clean up whitespace
            self.content = re.sub(r'\s+', ' ', text).strip()
            return True

        except Exception as e:
            print(f"Error fetching content from {self.href}: {str(e)}")
            return False

    def get_chunks(self, max_chunk_size: int = 1000) -> List[str]:
        """Split content into reasonably sized chunks for processing"""
        if not self.content:
            return [self.body]

        # Use sentence tokenization to create sensible chunks
        sentences = sent_tokenize(self.content)
        chunks = []
        current_chunk = ""

        for sentence in sentences:
            if len(current_chunk) + len(sentence) <= max_chunk_size:
                current_chunk += sentence + " "
            else:
                chunks.append(current_chunk.strip())
                current_chunk = sentence + " "

        if current_chunk:
            chunks.append(current_chunk.strip())

        return chunks if chunks else [self.body]


def search_web(query: str, max_results: int = TOP_SEARCH_RESULTS) -> List[SearchResult]:
    """
    Enhanced search function that prioritizes trusted domains and fetches content
    """
    ddgs = DDGS()

    # Add time filter to get more recent results
    time_restricted_query = f"{query} (after:2022)"

    # Get search results
    raw_results = list(ddgs.text(time_restricted_query, max_results=max_results*2))

    if not raw_results:
        return []

    # Convert to SearchResult objects
    results = [SearchResult(title=r['title'], body=r['body'], href=r['href']) for r in raw_results]

    # Prioritize trusted domains
    trusted_results = []
    other_results = []

    for result in results:
        is_trusted = any(domain in result.href for domain in TRUSTED_DOMAINS)
        if is_trusted:
            trusted_results.append(result)
        else:
            other_results.append(result)

    # Combine trusted and other results, prioritizing trusted ones
    final_results = trusted_results + other_results
    final_results = final_results[:max_results]

    # Fetch content for each result (in parallel in a production environment)
    for result in final_results:
        result.fetch_content()

    return final_results


In [5]:
# Enhanced web search module
from duckduckgo_search import DDGS
import requests
from bs4 import BeautifulSoup

@dataclass
class SearchResult:
    """Data class to store search result information"""
    title: str
    body: str
    href: str
    date: Optional[str] = None
    content: str = ""

    def __post_init__(self):
        # Clean up text
        self.title = re.sub(r'\s+', ' ', self.title).strip()
        self.body = re.sub(r'\s+', ' ', self.body).strip()

    def fetch_content(self, timeout: int = 10) -> bool:
        """Fetch the full content of the article"""
        try:
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
            }
            response = requests.get(self.href, headers=headers, timeout=timeout)
            response.raise_for_status()

            soup = BeautifulSoup(response.text, 'html.parser')

            # Remove script, style elements and comments
            for element in soup(['script', 'style', 'header', 'footer', 'nav']):
                element.decompose()

            # Extract main content based on common article containers
            article = soup.find('article') or soup.find(class_=re.compile(r'article|content|main|post'))

            if article:
                text = article.get_text(separator=' ', strip=True)
            else:
                # Fallback to main content area
                main = soup.find('main') or soup.find(attrs={'role': 'main'})
                if main:
                    text = main.get_text(separator=' ', strip=True)
                else:
                    # Last resort: get body text
                    text = soup.body.get_text(separator=' ', strip=True)

            # Clean up whitespace
            self.content = re.sub(r'\s+', ' ', text).strip()
            return True

        except Exception as e:
            print(f"Error fetching content from {self.href}: {str(e)}")
            return False

    def get_chunks(self, max_chunk_size: int = 1000) -> List[str]:
        """Split content into reasonably sized chunks for processing"""
        if not self.content:
            return [self.body]

        # Use sentence tokenization to create sensible chunks
        sentences = sent_tokenize(self.content)
        chunks = []
        current_chunk = ""

        for sentence in sentences:
            if len(current_chunk) + len(sentence) <= max_chunk_size:
                current_chunk += sentence + " "
            else:
                chunks.append(current_chunk.strip())
                current_chunk = sentence + " "

        if current_chunk:
            chunks.append(current_chunk.strip())

        return chunks if chunks else [self.body]


def search_web(query: str, max_results: int = TOP_SEARCH_RESULTS) -> List[SearchResult]:
    """
    Enhanced search function that prioritizes trusted domains and fetches content
    """
    ddgs = DDGS()

    # Add time filter to get more recent results
    time_restricted_query = f"{query} (after:2022)"

    # Get search results
    raw_results = list(ddgs.text(time_restricted_query, max_results=max_results*2))

    if not raw_results:
        return []

    # Convert to SearchResult objects
    results = [SearchResult(title=r['title'], body=r['body'], href=r['href']) for r in raw_results]

    # Prioritize trusted domains
    trusted_results = []
    other_results = []

    for result in results:
        is_trusted = any(domain in result.href for domain in TRUSTED_DOMAINS)
        if is_trusted:
            trusted_results.append(result)
        else:
            other_results.append(result)

    # Combine trusted and other results, prioritizing trusted ones
    final_results = trusted_results + other_results
    final_results = final_results[:max_results]

    # Fetch content for each result (in parallel in a production environment)
    for result in final_results:
        result.fetch_content()

    return final_results


In [6]:
def count_tokens(text: str) -> int:
    """Count the number of tokens in a text string"""
    return len(encoding.encode(text))

def truncate_to_token_limit(text: str, limit: int) -> str:
    """Truncate text to fit within a token limit"""
    tokens = encoding.encode(text)
    if len(tokens) <= limit:
        return text

    return encoding.decode(tokens[:limit])

def prepare_document_context(search_results: List[SearchResult], token_limit: int = MAX_TOKENS_CONTEXT) -> str:
    """
    Prepare the search results into a context document for the LLM,
    ensuring it doesn't exceed the token limit
    """
    context_parts = []
    current_tokens = 0

    # Add metadata and introduction
    metadata = f"Search results retrieved on {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n"
    current_tokens += count_tokens(metadata)

    for i, result in enumerate(search_results, 1):
        # Create article heading with source info
        article_heading = f"## Article {i}: {result.title}\nSource: {result.href}\n\n"
        heading_tokens = count_tokens(article_heading)

        # If we have the full content, use it; otherwise fall back to the snippet
        content = result.content if result.content else result.body
        content_tokens = count_tokens(content)

        # Check if adding this content would exceed our token limit
        if current_tokens + heading_tokens + content_tokens > token_limit:
            # If too long, truncate the content to fit
            available_tokens = token_limit - (current_tokens + heading_tokens)
            if available_tokens > 100:  # Only add if we can include meaningful content
                truncated_content = truncate_to_token_limit(content, available_tokens)
                context_parts.append(article_heading + truncated_content)
            break

        # Add the full content
        context_parts.append(article_heading + content)
        current_tokens += heading_tokens + content_tokens

    return metadata + "\n".join(context_parts)


In [7]:
def summarize_content(context: str, query: str, api_key: str) -> str:
    """
    Generate a comprehensive summary of the content using the OpenAI API
    """
    client = OpenAI(api_key=api_key)

    # Create a system prompt that encourages comprehensive and accurate summaries
    system_prompt = """You are an expert research assistant. Your task is to analyze and summarize the provided articles
    on a specific topic. Create a comprehensive, accurate, and balanced summary that:

    1. Captures the key findings, developments, and perspectives from all articles
    2. Identifies points of consensus and disagreement between sources
    3. Highlights limitations or gaps in the current research/information
    4. Organizes the information logically with clear section headers
    5. Includes relevant statistics, examples, or evidence when available
    6. Avoids inserting your own opinions or information not present in the articles
    7. Clearly distinguishes between well-established facts and claims/hypotheses

    Your summary should be detailed enough to serve as a foundation for answering follow-up questions.
    """

    user_prompt = f"""Topic: {query}

    Please summarize the following articles:

    {context}

    Create a comprehensive summary that covers all major aspects of this topic mentioned in the articles.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o",  # Using GPT-4o for better quality summaries
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.3,  # Lower temperature for more focused and factual summaries
            max_tokens=1500,  # Limit the summary length
        )

        return response.choices[0].message.content

    except Exception as e:
        return f"Error generating summary: {str(e)}"


In [8]:
def answer_follow_up_question(question: str, context: str, api_key: str) -> str:
    """
    Answer a follow-up question using retrieval-augmented generation
    """
    client = OpenAI(api_key=api_key)

    system_prompt = """You are an AI research assistant that answers questions based solely on the provided context.
    Follow these guidelines strictly:

    1. Only use information explicitly stated in the provided context
    2. If the answer isn't in the context, say "I don't have enough information to answer that question based on the provided articles"
    3. Do not introduce external knowledge or make assumptions beyond what's in the context
    4. Cite your sources by referring to specific articles in your answer (e.g., "According to Article 2...")
    5. When articles disagree on a point, present multiple perspectives and note the disagreement
    6. Be precise and specific in your answers
    7. If relevant, note limitations or uncertainties mentioned in the articles
    8. Structure complex answers with clear organization, using bullet points or numbered lists when appropriate

    Your goal is to provide accurate, helpful information without hallucinating or introducing external knowledge.
    """

    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
            ],
            temperature=0.2,  # Lower temperature to minimize hallucination
        )

        return response.choices[0].message.content

    except Exception as e:
        return f"Error answering question: {str(e)}"


In [9]:
def run_research_assistant(query: str, api_key: str) -> Tuple[str, List[SearchResult], str]:
    """
    Main function to run the research assistant workflow
    """
    try:
        # 1. Search for relevant articles
        search_results = search_web(query)

        if not search_results:
            return "No search results found. Please try a different query.", [], ""

        # 2. Prepare search results context
        context = prepare_document_context(search_results)

        # 3. Generate summary
        summary = summarize_content(context, query, api_key)

        # 4. Format sources
        sources = "## Sources\n\n"
        for i, result in enumerate(search_results, 1):
            sources += f"{i}. [{result.title}]({result.href})\n"

        # 5. Combine summary and sources
        full_response = f"{summary}\n\n{sources}"

        return full_response, search_results, context

    except Exception as e:
        return f"An error occurred: {str(e)}", [], ""


def handle_follow_up_question(question: str, context: str, api_key: str) -> str:
    """
    Handle a follow-up question using the RAG context
    """
    if not context:
        return "Please run a search first before asking follow-up questions."

    return answer_follow_up_question(question, context, api_key)


In [14]:
def create_gradio_interface():
    """
    Create a Gradio interface for the research assistant
    """
    with gr.Blocks(theme=gr.themes.Soft()) as app:
        gr.Markdown("""
        # 🔍 Advanced AI Research Assistant

        This tool searches the web for information on your topic, summarizes the findings, and answers your follow-up questions.

        ## How to use:
        1. Enter your research topic
        2. Click "Research" to search and summarize
        3. Ask follow-up questions about the research
        """)

        # Store API key and context
        api_key = gr.State(value=userdata.get('OpenAI'))
        context_state = gr.State(value="")
        search_results_state = gr.State(value=[])

        with gr.Row():
            with gr.Column():
                query = gr.Textbox(
                    label="Research Topic",
                    placeholder="Enter a research topic (e.g., 'Recent advances in quantum computing')",
                    lines=2
                )
                research_button = gr.Button("Research", variant="primary")

        with gr.Row():
            summary_output = gr.Markdown(label="Research Summary")

        with gr.Row():
            with gr.Column():
                follow_up = gr.Textbox(
                    label="Follow-up Question",
                    placeholder="Ask a specific question about the research results...",
                    lines=2
                )
                answer_button = gr.Button("Answer Question", variant="secondary")

        with gr.Row():
            answer_output = gr.Markdown(label="Answer")

        clear_button = gr.Button("Clear All")

        # Define event handlers
        def clear_all():
            return "", "", "", ""

        research_button.click(
            fn=run_research_assistant,
            inputs=[query, api_key],
            outputs=[summary_output, search_results_state, context_state]
        )

        answer_button.click(
            fn=handle_follow_up_question,
            inputs=[follow_up, context_state, api_key],
            outputs=answer_output
        )

        clear_button.click(
            fn=clear_all,
            inputs=[],
            outputs=[query, summary_output, follow_up, answer_output],
        )

    return app


In [ ]:
from openai.error import RateLimitError
import backoff

@backoff.on_exception(backoff.expo, RateLimitError)
def completions_with_backoff(**kwargs):
    response = client.chat.completions.create(**kwargs)
    return response


In [ ]:
if __name__ == "__main__":
    # Ensure OpenAI API key is available
    api_key = userdata.get('OpenAI')
    if not api_key:
        print("Error: OpenAI not found in Google Colab secrets.")
        print("Please add your OpenAI API key to the Colab secrets with the name 'OPENAI_API_KEY'.")
    else:
        # Launch Gradio interface
        app = create_gradio_interface()
        app.launch(debug=True)


Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a149269e25f958aa95.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Error fetching content from https://www.gartner.com/en/articles/what-s-new-in-artificial-intelligence-from-the-2022-gartner-hype-cycle: 403 Client Error: Forbidden for url: https://www.gartner.com/en/articles/what-s-new-in-artificial-intelligence-from-the-2022-gartner-hype-cycle
Error fetching content from https://www.gartner.com/en/articles/what-s-new-in-artificial-intelligence-from-the-2022-gartner-hype-cycle: 403 Client Error: Forbidden for url: https://www.gartner.com/en/articles/what-s-new-in-artificial-intelligence-from-the-2022-gartner-hype-cycle
